In [ ]:

import os
import sys
sys.path.append('./FEMGeoPackage')
sys.path.append(r"D:\GGW\IsoEmb")

from ngsolve import *
import netgen.gui
import time
from netgen.occ import *
from math import sin, cos, pi
import netgen.meshing as ngm
import numpy as np
from netgen.csg import *
from netgen.meshing import MeshingStep
from Package_ALE_Geometry import Vtk_out_BND
from ngsolve.comp import IntegrationRuleSpaceSurface
from Package_MyCode import FO


from geometry import *
from ngsolve.webgui import Draw
from ngsolve import Mesh
from netgen.occ import SplineApproximation, Pnt, Axis, Face, Wire, Segment, Revolve, OCCGeometry, Z, X, Y
from geometry.param_curve import phi, psi
import sympy as sp




# 输入参数都在这里
Ax, Ay, Az = 2, 2, 3  # 椭球在 x/y/z 方向的半轴；当前绕 z 轴旋转的网格构造要求 Ax=Ay
# [修改 P1] Example 6.1 主实验：tau=0.05, T=1；批量 h 列表见 ConvergenceK_paper.ipynb。
h_values = [0.7, 0.6, 0.5, 0.4, 0.3, 0.25]
tau = 5e-2; T = 1  # 固定时间步长、终止时刻；当前 BDF3 要求 T/tau 为整数且 >=3
h = 0.25; r = 6; k = 6;  # h: GenerateMesh 的 maxh；r: Regge 度量阶数；k: 数值解 H1 阶数
metric_bonus_intorder = 10  # gMh.Set 使用的额外求积阶数：在默认阶数上增加，不改变 Regge 阶数 r
#  NGSolve 中 order=20 的积分点空间返回直接指定 40 阶的三角形规则。
quad_order_ref = 20  # Cell 2 会核对实际规则，避免将此参数误读为求积阶数
cp_newton_iterations = 5  # 精确最近点 CF 的 Newton 次数；Cell 2 有独立精度检查
quad_order_asm = 30;num_threads_asm = 4
ds_asm = ds(intrules={
    ET.TRIG: IntegrationRule(
        ET.TRIG,
        order=quad_order_asm,
    )
})



# Mesh
# 生成旋转体 和 网格
def BaseTorus(phi): 
    res = Pnt(0, Ay*np.sin(phi),  Az*np.cos(phi))
    # res = Pnt(0, np.sin(phi),  np.cos(phi))
    return res
def GetRotFace(CurveFunc,T_min=0,T_max=np.pi,n=200):
    pnts = [CurveFunc(phi) for phi in np.linspace(T_min,T_max,n)]
    spline = SplineApproximation(pnts, tol=1e-8)
    f = Face(Wire([spline]))
    return f
f = GetRotFace(BaseTorus,T_min=0,T_max=np.pi)
torus = f.Revolve(Axis((0,0,0), Z), 360)
mesh = Mesh(OCCGeometry(torus).GenerateMesh(maxh=h,
            perfstepsend=ngm.MeshingStep.MESHSURFACE,grading=0.5))
Draw(mesh)


# FE Space
# r: 度量 Regge 阶数；k: 数值解及初始 Lagrange 插值的阶数。
fesG = HCurlCurl(mesh, order=r);  fesC = NumberSpace(mesh)
fesV = VectorH1(mesh,order=k); fes = H1(mesh,order=k);
fesMix = fesV*fesC*fesC*fesC*fesC*fesC*fesC


# Time step
# 当前 BDF3 公式使用固定步长；终点必须落在时间网格上。
if not (np.isfinite(tau) and np.isfinite(T) and tau > 0 and T > 0):
    raise ValueError("tau 和 T 必须是有限正数")
N_steps = int(round(T / tau))
if not np.isclose(N_steps * tau, T, rtol=1e-12, atol=0.0):
    raise ValueError("固定步长 BDF3 要求 T/tau 为整数；请设置 tau = T/N_steps")
if N_steps < 3:
    raise ValueError("BDF3 使用 0、tau、2*tau 的初值，要求 T 至少为 3*tau")
tau = T / N_steps  # 仅统一可接受的浮点舍入差异
told = 0.0

folder_name = f"./RateRe/ell_paper/h_{h:.2e}_tau_{tau:.2e}_T_{T:.2e}_r_{r}_k_{k}"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
output_path = folder_name




In [ ]:
# Cell 2



# NGSolve 原生积分点空间：每个自由度保存一个曲面积分点上的标量值。
surface_region = mesh.Boundaries('.*')  # 三维网格的全部表面；'.*' 是匹配所有边界名称的正则表达式
fesir = IntegrationRuleSpaceSurface(
    mesh, order=quad_order_ref, definedon=surface_region
)
irs = fesir.GetIntegrationRules()  # 各单元类型对应的积分点与权重；后续 ds(intrules=irs) 必须共用它
# [修改 P1] 核实误差求积确为论文指定的 40 阶规则。
if str(irs[ET.TRIG]) != str(IntegrationRule(ET.TRIG, order=40)):
    raise RuntimeError('积分点空间规则不是预期的 40 阶规则，请核对 NGSolve 版本与 quad_order_ref')
fesirV = fesir**3  # 三个标量积分点空间的乘积，用来保存三维向量，不是三阶 H1 空间
X_ref0 = GridFunction(fesirV)
X_ref1 = GridFunction(fesirV)
# 在同一组积分点上直接求值坐标函数 (x,y,z)；
coords_ir = GridFunction(fesirV)
coords_ir.Interpolate(CF((x, y, z)), surface_region)
coords_3d = np.column_stack([component.vec.FV().NumPy() for component in coords_ir.components])
del coords_ir  # NumPy 坐标数组已独立保存，释放临时积分点函数
# 把积分点坐标 coords_3d 投影到 surface 上 --> points
# 当前 X_perturbed 为轴对齐椭球；直接求最近点，避免采样树和参数极点退化。
from ellipsoid_projection import project_ellipsoid
phi_values, psi_values, points, tol_info = project_ellipsoid(
    coords_3d, (Ax, Ay, Az), tol=1e-15, max_iter=50
)
# 返回信息：同时检查曲面方程和法向投影条件。
print(f"projection: {len(points)} points, {tol_info['iterations']} iterations, "
      f"{tol_info['elapsed_seconds']:.3f} s; "
      f"surface residual={tol_info['max_surface_residual']:.3e}, "
      f"normal residual={tol_info['max_normal_residual']:.3e}")
if tol_info['final_delta_norms'].size:
    idx = tol_info['non_converged_indices']
    print(list(idx))
    print("max remaining projection step:", float(np.max(tol_info['final_delta_norms'])))
    raise RuntimeError("椭球投影未达到容差，请检查 tol_info；停止使用未收敛的参考点")
else:
    print("all points converged within the tolerance")
# 利用 points 完成 nodal interpolation for IntRuleSpace
def_coords = points; 
vec = def_coords[:,0]; 
X_ref0.components[0].vec.data = BaseVector(vec)
X_ref1.components[0].vec.data = BaseVector(1/2*vec)
vec = def_coords[:,1];
X_ref0.components[1].vec.data = BaseVector(vec)
X_ref1.components[1].vec.data = BaseVector(1/2*vec)
vec = def_coords[:,2]; 
X_ref0.components[2].vec.data = BaseVector(vec)
X_ref1.components[2].vec.data = BaseVector(1/3*vec)




# [修改 P2] 可在任意 Regge 边/面求积点求值的精确最近点映射及解析 Jacobian。
# 保留上面的 NumPy 最近点计算用于最终误差；不把积分点 GridFunction 用于 Regge Set。
from isoemb_paper_geometry import (
    ellipsoid_cp_jacobian_cf, check_geometry_cf, interpolate_surface_lagrange,
)
a_geometry, Da_geometry = ellipsoid_cp_jacobian_cf(
    (Ax, Ay, Az), iterations=cp_newton_iterations,
)
geometry_check = check_geometry_cf(
    a_geometry, Da_geometry, fesir, coords_3d, points, (Ax, Ay, Az),
)
print(f"exact geometry check: point={geometry_check['point']:.3e}, "
      f"Jacobian={geometry_check['jacobian']:.3e}")


In [ ]:
# Cell 3

# [修改 P3] 1. 真正的 Lagrange 节点插值：Xh0 = I_h(r(0) o a)。
# 在三角形等距重心节点求精确值，再转换为 NGSolve 分层基底系数；替代全局 L2 投影。
mesh.UnsetDeformation()  # 允许执行可视化后重新初始化
def exact_initial_values(node_coords):
    _, _, node_points, node_info = project_ellipsoid(
        node_coords, (Ax, Ay, Az), tol=1e-15, max_iter=50,
    )
    if node_info['non_converged_indices'].size:
        raise RuntimeError("Lagrange 节点最近点投影未收敛")
    return node_points

initial_coeffs, initial_interp_info = interpolate_surface_lagrange(
    fes, exact_initial_values, k,
)
Xh0 = GridFunction(fesV); Xh1 = GridFunction(fesV)
for i, factor in enumerate((1/2, 1/2, 1/3)):
    Xh0.components[i].vec.FV().NumPy()[:] = initial_coeffs[:, i]
    # r(1) 是 r(0) 各分量的常数倍，I_h 与该缩放可交换。
    Xh1.components[i].vec.data = factor * Xh0.components[i].vec
Exa_ptr = GridFunction(fesV)
Exa_ptr.vec.data = Xh1.vec - Xh0.vec  # 仅为插值参考速度，右端使用精确导数
print(f"Lagrange initial data: max nodal error={initial_interp_info['max_nodal_error']:.3e}")


# [修改 P4] 2. 论文式 (3.8) 的固定背景：gMh = Rh(a* g_M)。
nsurf = specialcf.normal(3); NN = OuterProduct(nsurf, nsurf); Ptau = Id(3) - NN
J0_geometry = Da_geometry * Ptau  # 任意求积点上的精确 dr(0) 拉回
J = J0_geometry; G = J.trans * J  # 不再对 Xh0 求导来生成背景度量
gMh = GridFunction(fesG)
gMh.Set(
    G, definedon=mesh.Boundaries('.*'),
    dual=True,
    bonus_intorder=metric_bonus_intorder,
)
g_inv = Ptau * Inv(gMh + NN) * Ptau
sq_det = sqrt(Det(gMh + NN))


# [修改 P3] 3. 论文的三个启动值 I_h(r(t_j) o a)，直接组合系数，j=0,1,2。
Pos0 = GridFunction(fesV);Pos1 = GridFunction(fesV); 
Pos_1 = GridFunction(fesV);Pos_2 = GridFunction(fesV); 
Pos_hat = GridFunction(fesV)



Solution= GridFunction(fesMix);
SetVisualization(deformation=True)

told = 0.0
Pos_2.vec.data = (1 - told) * Xh0.vec + told * Xh1.vec
told = tau
Pos_1.vec.data = (1 - told) * Xh0.vec + told * Xh1.vec
told = 2 * tau
step_n = 2  # 当前历史解对应的整数时间步
Pos0.vec.data = (1 - told) * Xh0.vec + told * Xh1.vec





In [ ]:
# Cell 4

# [修改 P5] 仍用原优化后的 BDF3 主体，仅改右端数据来源和每步诊断标签。
%run -i IsoEmbRate_cell4_paper.py



In [ ]:
# Cell 5

# 最终汇总：先运行时间推进单元和 Cell 5 的 err 计算。
if step_n != N_steps or not np.isclose(told, T, rtol=1e-12, atol=0.0):
    raise RuntimeError("请先将时间推进到 T，再计算终点度量误差")

# 在原始离散网格 Mh 上计算；允许运行可视化单元后再次执行本单元。
mesh.UnsetDeformation()

# 1. L2、strain、graph 误差沿用原 notebook 的精确曲面定义。
# 最近点投影 a(x) 的解析 Jacobian
axes2 = np.array([Ax, Ay, Az], dtype=float)**2
cp_normal = points / axes2
lam_cp = (
    np.sum((coords_3d - points) * cp_normal, axis=1)
    / np.sum(cp_normal**2, axis=1)
)
diag_B = 1.0 / (1.0 + lam_cp[:, None] / axes2)
Bn = diag_B * cp_normal
Da_values = (
    diag_B[:, :, None] * np.eye(3)
    - Bn[:, :, None] * Bn[:, None, :]
    / np.sum(cp_normal * Bn, axis=1)[:, None, None]
)
# 将 Jacobian 的九个分量存入同一个积分点空间
Da_entries = []
for i in range(3):
    for j in range(3):
        entry = GridFunction(fesir)
        entry.vec.FV().NumPy()[:] = Da_values[:, i, j]
        Da_entries.append(entry)
Da_cf = CF(tuple(Da_entries), dims=(3, 3))

# 2. 精确参考解及误差的切向导数
nh = specialcf.normal(3);Nh = OuterProduct(nh, nh);Ph = Id(3) - Nh
J0_exact = Da_cf * Ph
Dt = CF((
    1-T/2, 0, 0,
    0, 1-T/2, 0,
    0, 0, 1-2*T/3,
), dims=(3, 3))
Jr_exact = Dt * J0_exact
Je = grad(Pos1).Trace() - Jr_exact
S = 0.5 * (Jr_exact.trans * Je + Je.trans * Jr_exact)
r_minus_l = (1-T)*X_ref0 + T*X_ref1
e = Pos1 - r_minus_l

# 3. 原始曲面背景度量的精确拉回
G0_exact = J0_exact.trans * J0_exact;H_exact = Ph * Inv(G0_exact + Nh) * Ph;mu_exact = sqrt(Det(G0_exact + Nh))
ds_err = ds(intrules=irs)

L2_sq = Integrate(
    InnerProduct(e, e) * mu_exact * ds_err, mesh
)
strain_sq = Integrate(
    Trace(H_exact*S*H_exact*S) * mu_exact * ds_err, mesh
)
err_graph = sqrt(L2_sq + strain_sq)

# [P7] metric error 的目标和范数均为精确曲面定义。
# 这与论文式 (6.3) 的离散 metric defect 有意不同，详见 Cell 6。
# 4. 精确目标度量 a^*g(T) = Jr_exact.trans * Jr_exact。
# 误差定义在精确参考曲面 M 上，计算时通过最近点投影拉回 Mh。
# 使用恒等式 Jh.T*Jh - Jexact.T*Jexact = 2*S + Je.T*Je，
# 避免高阶精度下两个 O(1) 度量直接相减产生的消减误差。
g_exact = Jr_exact.trans * Jr_exact
metric_defect = 2*S + Je.trans * Je

# t=0 使用实际初值 Xh0，与精确 r(0) 的导数 J0_exact 比较。
Je_initial = grad(Xh0).Trace() - J0_exact
S_initial = 0.5 * (
    J0_exact.trans * Je_initial + Je_initial.trans * J0_exact
)
initial_metric_defect = 2*S_initial + Je_initial.trans * Je_initial

# 5. 精确 M 上的张量 L2 范数；背景始终是固定的 g_M，而非 g(T)。
# H_exact、mu_exact 分别是 a^*g_M 的切向逆和面积权重。
# 与 graph error 共用 ds_err 及其积分点空间，不能混用 ds_asm。
initial_metric_sq = Integrate(
    Trace(H_exact*initial_metric_defect*H_exact*initial_metric_defect)
    * mu_exact * ds_err,
    mesh,
)
metric_sq = Integrate(
    Trace(H_exact*metric_defect*H_exact*metric_defect) * mu_exact * ds_err,
    mesh,
)
# 保留结果变量名，供 run_isoemb_convergence.py 读取和保存。
err_initial_metric = sqrt(initial_metric_sq)
err_metric = sqrt(metric_sq)

# [修改 P6] 同时区分嵌入自由度与包含六个乘子的系统总自由度。
ndof = fesV.ndof  # CSV 的既有字段仍为 3*dim(Vh)，生成表格时再加 6
ndof_total = fesMix.ndof
print(f"h: {h:.6e}, DoF (total): {ndof_total}, embedding dof: {ndof}, L2error: {sqrt(L2_sq):.12e}, strainerr: {sqrt(strain_sq):.12e}, grapherr: {err_graph:.12e}, metricerr: {err_metric:.12e}, initial metric error: {err_initial_metric:.12e}")



In [ ]:
# [修改 P8] 网格显示位置应为 Pos1：位移相对于原始网格坐标 (x,y,z)。
mesh.UnsetDeformation()  # 也允许重复执行本单元
deform = GridFunction(fesV)
deform.Set(Pos1 - CF((x, y, z)), definedon=surface_region)
mesh.SetDeformation(deform)
Draw(mesh)
